<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>LPU — BACKEND & GENERATIVE AI</b>

## Day 2 — Exercises: Pydantic Validation & Gradio UIs

**Lovely Professional University**
*Backend & Generative AI · Poorit Technologies*

</div>

---

### What You'll Practise

Five short tasks, then one **mini-project** you write yourself.

1. **Q1** — define a Pydantic model and build a valid object
2. **Q2** — catch a bad value with `ValidationError`
3. **Q3** — see why Pydantic catches what a `TypedDict` misses
4. **Q4** — build your first `gr.Interface`
5. **Q5** — build a chatbot with `gr.ChatInterface`
6. **Mini-project** — a **Company Pamphlet Generator**: scrape a landing page, let the model write the pamphlet, stream it into a UI

> **Q1–Q4 need no API key at all.** Only Q5 and the mini-project do.

---

## 1. Setup

Run these two cells first.

In [ ]:
# Install what we need
!pip install -q openai gradio pydantic requests beautifulsoup4

In [ ]:
import os
from getpass import getpass
from typing import TypedDict
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError
import gradio as gr
import requests
from bs4 import BeautifulSoup

# Q1-Q4 need no key. Press Enter to skip; you can re-run this cell before Q5.
openai_api_key = getpass("OpenAI API Key (press Enter to skip - only Q5 and the mini-project need it): ")

MODEL = "gpt-4o-mini"
openai_client = None
if openai_api_key:
    os.environ["OPENAI_API_KEY"] = openai_api_key
    openai_client = OpenAI()
    print("Ready - key loaded, everything in this notebook will work.")
else:
    print("Ready - no key. Q1 to Q4 will work; re-run this cell before Q5.")

---

## 2. Pydantic — data that checks itself

A Pydantic model is a class that **describes the shape of your data and enforces it**.
Build one with bad data and it refuses, loudly, instead of letting the mistake travel
downstream.

### Q1: Define a model and build one

Write the field types, then build a valid `Student`.

In [ ]:
# Hint: name is text (str), roll is a whole number (int).
# Field(ge=..., le=...) sets a minimum and a maximum - a CGPA is out of 10.

class Student(BaseModel):
    name: ___
    roll: ___
    cgpa: float = Field(ge=0, le=___)


aditi = Student(name="Aditi", roll=42, cgpa=8.7)
print(aditi)
print("Her CGPA is:", aditi.cgpa)

### Q2: Catch a bad value

A CGPA of 99 breaks the `le=10` rule. Make it fail, and catch the error.

In [ ]:
# Hint: which exception does Pydantic raise when validation fails?
# You imported it at the top of this notebook.

try:
    rahul = Student(name="Rahul", roll=43, cgpa=___)     # out of range on purpose
    print(rahul)
except ___ as e:
    print("Rejected — and here is exactly why:\n")
    print(e)

### Q3: Pydantic vs `TypedDict`

A `TypedDict` looks like it does the same job. Put **clearly wrong** data into both
and see which one actually complains.

In [ ]:
# Hint: put text where a number belongs - e.g. roll="not-a-number", cgpa="excellent".

class StudentDict(TypedDict):
    name: str
    roll: int
    cgpa: float


# 1) The TypedDict version
sloppy = StudentDict(name="Meera", roll="not-a-number", cgpa="excellent")
print("TypedDict accepted it:", sloppy)

# 2) The Pydantic version - same bad data
try:
    Student(name="Meera", roll=___, cgpa=___)
except ValidationError as e:
    print("\nPydantic rejected it:\n")
    print(e)

---

## 3. Gradio — turn a function into a web app

`gr.Interface` wraps **any Python function** in a UI. In Colab it renders right below
the cell; add `share=True` inside `launch()` for a public link you can open on your phone.

**No starter code from here on — you write it.**

### Q4: Your first Interface

Build a **word counter** as a web app: someone types a sentence, the app answers
`"5 words, 28 characters"`.

Write the counting function yourself, then wrap it in a `gr.Interface` — a textbox in,
a textbox out — and launch it. No AI here, just a plain Python function with a UI on top.

**Tip:** print the function's output once before you build the UI. Far easier to debug.

In [ ]:
# Your code here.



### Q5: A chatbot with a personality

Build a chatbot with `gr.ChatInterface` that answers everything in **pirate slang** —
or pick your own personality: strict teacher, cricket commentator, whatever you like.

Your function gets two things: the new `message`, and the `history` so far. With
`type="messages"`, that history is already a list of `{"role": ..., "content": ...}`
dicts — the exact shape the API wants.

**Needs your API key** — re-run the setup cell if you skipped it.

In [ ]:
# Your code here.



---

## 4. Mini-project — Company Pamphlet Generator

**What you're building:** type a company name and its website URL → your app **fetches the
landing page**, hands that text to the model, and **streams a short pamphlet** (for
prospective customers, investors and recruits) into a Gradio UI.

```
name + URL  →  scrape the page  →  model writes the pamphlet  →  streams into the UI
```

Steps 1 and 2 are given — run them. The rest is yours.

> **Needs your API key** — re-run the setup cell if you pressed Enter past it.

In [ ]:
# Step 1 - The scraper. PROVIDED: just run this cell.

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url, max_chars=2_000):
    """Return the title and visible text of the page at `url`."""
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(response.content, "html.parser")

        title = soup.title.string if soup.title else "No title found"

        if soup.body:
            for tag in soup.body(["script", "style", "img", "input"]):
                tag.decompose()                                  # drop the noise
            text = soup.body.get_text(separator="\n", strip=True)
        else:
            text = ""

        return (title + "\n\n" + text)[:max_chars]
    except Exception as e:
        return f"Error fetching website: {e}"


print("Scraper ready.")

In [ ]:
# Step 2 - Test it first. PROVIDED: run it, then change the URL and re-run.

page = fetch_website_contents("https://anthropic.com")
print(page[:300])

In [ ]:
# Step 3 - Write `pamphlet_system`: ask for a short pamphlet, in markdown,
# using only the page text.



In [ ]:
# Step 4 - Pick one company: build the prompt (name + scraped text) and print it.



In [ ]:
# Step 5 - Write stream_pamphlet(company_name, url): call the model with
# stream=True and `yield` the text as it grows.



In [ ]:
# Step 6 - Put it in a gr.Interface (name + URL in, markdown out) and launch it.



In [ ]:
# Step 7 (stretch) - Add a tone dropdown: professional / playful / recruiter-facing.



**When it misbehaves:**

| What you see | What it means |
|---|---|
| `Error fetching website: ...` | the site blocked the request, or the URL is missing `https://` — try another company |
| The pamphlet appears all at once | your function `return`s instead of `yield`ing — a streaming UI needs a generator |
| Facts that are nowhere on the page | the model filled the gaps itself — 2,000 characters is a small window |
| Blank UI / the cell is stuck | you skipped the API key. Re-run the setup cell, then re-run Step 6 |

---

### ✅ What you practised

| Idea | The one-liner |
|---|---|
| **Pydantic model** | a class that describes your data's shape — and enforces it |
| **`Field(ge=, le=)`** | a range check the model applies for you |
| **`ValidationError`** | raised the moment bad data appears, not three functions later |
| **`TypedDict`** | hints only — at runtime it is just a dict, and it accepts anything |
| **`gr.Interface`** | any Python function → a web UI |
| **`gr.ChatInterface`** | a chatbot, where `history` gives it memory for free |
| **Scrape → prompt** | your code fetches the page; the model only ever sees text you handed it |
| **`yield` in a UI function** | every yield repaints the output box, so the answer types out live |

**Finished early?** Try these:
1. Add an `email` field to `Student` and reject anything without an `@`.
2. Give the counter Interface a second output box that also shows the longest word.
3. Add `share=True` to `launch()` and open your chatbot on your phone.
4. Point the pamphlet generator at your own college — then tune the system message until
   the output reads like something you would actually hand to someone.